# DiscoRL PyTorch meta-training skeleton

This notebook shows the JAX-free PyTorch training path and can be extended with outer-loop meta-optimization.

In [ ]:
import torch
from ml_collections import config_dict
from disco_rl.agent import Agent
from disco_rl.environments.catch import CatchEnvironment, get_config

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)

In [ ]:
env = CatchEnvironment(batch_size=8, env_settings=get_config())
settings = config_dict.ConfigDict({
    "update_rule_name": "disco",
    "update_rule": {"net": {"name": "lstm", "prediction_size": 32, "hidden_size": 128}, "value_discount": 0.995, "max_abs_value": 10.0, "num_bins": 51},
    "net_settings": {"name": "mlp", "net_args": {"dense": (64, 64), "prediction_size": 32, "num_bins": 51}},
    "learning_rate": 3e-4,
    "max_abs_update": 1.0,
    "hyper_params": {"pi_cost": 1.0, "y_cost": 0.1, "z_cost": 0.1, "aux_policy_cost": 0.1, "value_cost": 0.5, "target_params_coeff": 0.995},
})
agent = Agent(single_observation_spec=env.observation_spec(), single_action_spec=env.action_spec(), agent_settings=settings, device=device)

In [ ]:
# Collect fixed-length rollouts, then call agent.update(...).
# The meta-network is an nn.Module (agent.update_rule.meta_net) and can be optimized by an outer-loop objective.
print(agent.network)